# Exploratory Data Analysis Organized

In [1]:
print('Kernel test')

Kernel test


In [2]:
# Step 0, lets import the regular stuff we will probably need
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
summarize)
print('Done part 1212323')

# MORE MOUSE BITES
# Ehh, we should really start learning SCIKITLEARN next or whatever its called
# but I think for this project its not horrible to stick with the ISLP resources
# since we ARE straying from the notes now
from ISLP import confusion_table
from ISLP.models import contrast
from sklearn.discriminant_analysis import \
(LinearDiscriminantAnalysis as LDA ,
QuadraticDiscriminantAnalysis as QDA)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import re
print('Done part 2 asddfsdf')

Done part 1212323
Done part 2 asddfsdf


In [7]:
# Import the data:
Titanic_train = pd.read_csv('titanic_data/train.csv')

Titanic_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Here is the summary of our EDA.

<img src="images_titanic/EDA summary.png" width="1300">

The plan is this:

 1(a)Drop 'PassengerId' column

 1(b) Create column called 'Title' to estimate the missing ages. Estimate missing ages

 1(c) Combine SibSp and ParCh columns into one (SibSp + ParCh). Then make it into bins (we dont want bins that are too small, to avoid overfitting). Drop 'SibSP', 'ParCh' columns as well as any created temporarily to calculate this Family_Bins style column

 1(d) Make 'Has_Cabin_Record' column. Drop 'Cabin' column 

 1(e) Use ticket to calculate log+1(Fare). Create 'log_plus1_Fare' column. Drop 'Fare' and 'Ticket' columns.

 1(f) Delete the rows where 'Embarked' is null. Setup so that for future training, we assume Embarked = Mode (S?)

In [8]:
#1(a)

# We drop the useless looking column
Titanic_train = Titanic_train.drop(columns='PassengerId')

Titanic_train.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [10]:
# 1(b)

print('The median age  was ' + str(Titanic_train['Age'].median()))

# Then we use regex to find the different groups, and make a new column. The column we care about is 'Name', we categorize according to cat we found and what google says they mean
conditions_name = [
    Titanic_train['Name'].str.contains(r'Master') 
    , Titanic_train['Name'].str.contains(r'Mrs.') | Titanic_train['Name'].str.contains(r'Mme.')# Married woman Mme
    , Titanic_train['Name'].str.contains(r'Mr.') # Mr.
    , Titanic_train['Name'].str.contains(r'Miss.') | Titanic_train['Name'].str.contains(r'Mlle.') # Girl or unmarried woman, Unmarried
]

choices_name = ['Master'
                , 'Mrs/Mme'
                , 'Mr'
                , 'Miss/Mlle']

Titanic_train['Title'] = np.select(conditions_name, choices_name, default = 'Other')

# We review the value counts, Other is a BIT suspect in terms of being too small a category, but since we are using this for age estimation as opposed to category, I think this is ok
Titanic_train['Title'].value_counts()

Titanic_train['Age'] = Titanic_train['Age'].fillna(
    Titanic_train.groupby('Title')['Age'].transform('median')
)

print('The median age has changed to be ' + str(Titanic_train['Age'].median()))

The median age  was 30.0
The median age has changed to be 30.0


In [11]:
# 1(c)
Titanic_train['Family'] = Titanic_train['SibSp']+Titanic_train['Parch']

# Create bins for SibSp and ParCh
bins_01_2 = [-0.5, 0.5, 1.5, 1000]
labels_01_2 = ['Zero', 'One', 'Two or more']

# Create bins for "Family"
bins_012_3 = [-0.5, 0.5, 1.5, 2.5, 1000]
labels_012_3 = ['Zero', 'One', 'Two', 'Three or more']

Titanic_train['Sibsp_bins'] = pd.cut(Titanic_train['SibSp'], bins_01_2, labels = labels_01_2)
Titanic_train['Parch_bins'] = pd.cut(Titanic_train['Parch'], bins_01_2, labels = labels_01_2)
Titanic_train['Family_bins'] = pd.cut(Titanic_train['Family'], bins_012_3, labels = labels_012_3)
Titanic_train.head(10)

# Drop actual values. Bins to be decided later
Titanic_train = Titanic_train.drop(columns='SibSp')
Titanic_train = Titanic_train.drop(columns='Parch')
Titanic_train = Titanic_train.drop(columns='Family')

In [12]:
# 1(d)
Titanic_train['Has_Cabin_Record'] = Titanic_train['Cabin'].notna().astype(int)

Titanic_train = Titanic_train.drop(columns='Cabin')

In [13]:
# 1(e)

ticket_counts = Titanic_train['Ticket'].value_counts()
group_size_by_ticket = Titanic_train['Ticket'].map(ticket_counts)

Titanic_train['Individual_Fare_LogPlus1'] = np.log1p(Titanic_train['Fare'] / group_size_by_ticket)

Titanic_train = Titanic_train.drop(columns='Fare')
Titanic_train = Titanic_train.drop(columns='Ticket')

In [ ]:
# 1(f)
Titanic_train = Titanic_train.dropna(subset=['Embarked'])

# 1(g) (For test data:) Fill missing 'Embarked' values in the test set with 'S'
#Titanic_test['Embarked'] = Titanic_test['Embarked'].fillna('S')

In [15]:
# Final checkup
Titanic_train

,Survived,Pclass,Name,Sex,Age,Embarked,Title,Sibsp_bins,Parch_bins,Family_bins,Has_Cabin_Record,Individual_Fare_LogPlus1
0,0,3,"Braund, Mr. Owen Harris",male,22.0,S,Mr,One,Zero,One,0,2.110213
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,C,Mrs/Mme,One,Zero,One,1,4.280593
2,1,3,"Heikkinen, Miss. Laina",female,26.0,S,Miss/Mlle,Zero,Zero,Zero,0,2.188856
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,S,Mrs/Mme,One,Zero,One,1,3.316003
4,0,3,"Allen, Mr. William Henry",male,35.0,S,Mr,Zero,Zero,Zero,0,2.202765
...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,"Montvila, Rev. Juozas",male,27.0,S,Other,Zero,Zero,Zero,0,2.639057
887,1,1,"Graham, Miss. Margaret Edith",female,19.0,S,Miss/Mlle,Zero,Zero,Zero,1,3.433987
888,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,21.0,S,Miss/Mlle,One,Two or more,Three or more,0,2.543569
889,1,1,"Behr, Mr. Karl Howell",male,26.0,C,Mr,Zero,Zero,Zero,1,3.433987
